In [21]:
import pandas as pd
import numpy
import requests
import json
import matplotlib.pyplot as plt
import os
from pathlib import Path
import sys


In [22]:
wka = pd.read_excel("../data/raw/Zeitreihen_WKA_2025_2026.xlsx")
weather = pd.read_excel("../data/raw/Zeitreihen_Wetterprognosen_2025_2026.xlsx")

In [23]:
wka.head(10)

,Größe,Unnamed: 1,kW el,kW el.1
0,Richtung,NaN,Einsp,Einsp
1,Zeitraster Min,NaN,15,15
2,Zeitverschiebung UTC,NaN,UTC+1,UTC+1
3,Lokale Zeit,NaN,Nein,Nein
4,Datum_von,Datum_bis,Unterpreilipp_Rudolstadt,Burgau_Jena
5,2025-01-01 00:00:00,01.01.2025 00:15:00,812,851.4
6,01.01.2025 00:15:00,01.01.2025 00:30:00,810,851.8
7,01.01.2025 00:30:00,01.01.2025 00:45:00,808,849.4
8,01.01.2025 00:45:00,01.01.2025 01:00:00,814,844.6
9,01.01.2025 01:00:00,01.01.2025 01:15:00,810,845


In [35]:
weather.head(10)

,Größe,Unnamed: 1,Globalstrahlung W/qm,Temperatur C,Windgeschwindigkeit m/s,Windrichtung
0,Zustand,NaN,Prognose,Prognose,Prognose,Prognose
1,Zeitraster Min,NaN,15,15,15,15
2,Zeitverschiebung UTC,NaN,UTC+1,UTC+1,UTC+1,UTC+1
3,Lokale Zeit,NaN,Nein,Nein,Nein,Nein
4,Datum_von,Datum_bis,Wert,Wert,Wert,Wert
5,01.01.2025 00:00:00,01.01.2025 00:15:00,0,0.7,8,222
6,01.01.2025 00:15:00,01.01.2025 00:30:00,0,0.7,8,222
7,01.01.2025 00:30:00,01.01.2025 00:45:00,0,0.7,8,222
8,01.01.2025 00:45:00,01.01.2025 01:00:00,0,0.7,8,222
9,01.01.2025 01:00:00,01.01.2025 01:15:00,0,-0.2,7,226


In [24]:

url = "https://kshww2.thueringen.de/FROST-Server/v1.1/Things"

response = requests.get(url, timeout=30)
data = response.json()

result = {
    thing["name"]: {
        "description": thing.get("description"),
        "properties": thing.get("properties", {})
    }
    for thing in data["value"]
}


In [25]:
result

{'4834900280': {'description': 'groundwater station (auto-importer)',
  'properties': {'creation_date': '2024-08-13T17:01:25.803000',
   'station_id': '4834900280',
   'type': 'AUTO_IMPORT_SENF'}},
 'Grundwasserstation@5031210655': {'description': 'groundwater station (auto-importer)',
  'properties': {'creation_date': '2024-08-13T17:02:04.725852',
   'station_id': '5031210655',
   'type': 'AUTO_IMPORT_SENF'}},
 'Grundwasserstation@5032210675': {'description': '',
  'properties': {'creation_date': '2024-11-19T13:21:27.821157',
   'station_id': '5032210675',
   'type': 'AUTO_IMPORT_SENF'}},
 'Bodenfeuchte_Messstation@121588': {'description': '',
  'properties': {'creation_date': '2024-11-19T13:48:31.251772',
   'station_id': '121588',
   'type': 'AUTO_IMPORT_SENF'}},
 'Bodenfeuchte_Messstation@121589': {'description': '',
  'properties': {'creation_date': '2024-11-19T16:27:15.585112',
   'station_id': '121589',
   'type': 'AUTO_IMPORT_SENF'}},
 'Bodenfeuchte_Messstation@121590': {'descr

In [26]:
locations = list(result.keys())

In [27]:
# hack
project_root = Path.cwd().parent  
sys.path.append(str(project_root))

In [28]:
#fetching the data from the water locations
from src.data.fetch_data import fetch_data_good

# default Q becase m³ / s is more useful than cm
default_param = "Q"
day_data = "1D"
min_data = "15min"
start_date = "2025-01-01"
end_date = "2026-01-01"

In [29]:
missing_stations = 0
# takes a while, uncomment for fetch
for station in locations:
    try:
        fetch_data_good(station, default_param, min_data, start_date, end_date, "../data/raw/water_data")
    except:
        missing_stations += 1

Station=4834900280  Parameter=Q  Freq=15min  2025-01-01T00:00:00Z → 2026-01-01T00:00:00Z

Error fetching No datastream for parameter='Q', freq='15min' on Thing 113. Run explore_api.py → list_datastreams(113) to see what is available.
Station=Grundwasserstation@5031210655  Parameter=Q  Freq=15min  2025-01-01T00:00:00Z → 2026-01-01T00:00:00Z

Error fetching No datastream for parameter='Q', freq='15min' on Thing 124. Run explore_api.py → list_datastreams(124) to see what is available.
Station=Grundwasserstation@5032210675  Parameter=Q  Freq=15min  2025-01-01T00:00:00Z → 2026-01-01T00:00:00Z

Error fetching No datastream for parameter='Q', freq='15min' on Thing 176. Run explore_api.py → list_datastreams(176) to see what is available.
Station=Bodenfeuchte_Messstation@121588  Parameter=Q  Freq=15min  2025-01-01T00:00:00Z → 2026-01-01T00:00:00Z

Error fetching No datastream for parameter='Q', freq='15min' on Thing 178. Run explore_api.py → list_datastreams(178) to see what is available.
Stati

In [34]:
print(f"Number of missing stations: {missing_stations}")

n_files = sum(
    1 for f in os.listdir("../data/raw/water_data")
    if os.path.isfile(os.path.join("../data/raw/water_data", f))
)

print(f"Number of successfull files: {n_files}")

Number of missing stations: 32
Number of successfull files: 138


### Exploring Market Price and reBAP Data

In [ ]:
market_price = pd.read_csv("../data/raw/Zeitreihen_Marktpreis_20180930_20260613_UTC.csv", sep=";")
rebap = pd.read_csv("../data/raw/reBAP unterdeckt [2026-06-13 09-51-13].csv", sep=";")
display(market_price.head())
display(rebap.head())

In [ ]:
print("Market Price Info:")
market_price.info()
print("\nReBAP Info:")
rebap.info()

In [ ]:
print("WKA Info:")
wka.info()
print("\nWeather Info:")
weather.info()